# Fire distribution across Australia

## Accessing Wildfire Data via API

In [2]:
# import necessary libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


In [3]:
# 1.
# access api url

## satellite: VIIRS SNPP NRT 
## area: 'world' = entire world 
## day range: '5' = data of the last 5 days (more doesnt wooooooork)
## date: None = most recent available data, so today's data

MAP_KEY = '4899a992545cbeb46f9fd0b6a025ef17'
area_url ='https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_SNPP_NRT/world/5' # warum gehen 1, 3 oder 5 tage aber ab 8 oder so nicht mehr??

# 2.
# read in the data from URL

df_area = pd.read_csv(area_url)

# 3.
# have a first glimpse at the data

df_area.head(5)
df_area.shape

(134201, 14)

## Cleaning and Rearranging Data

### Filter for Data only within Australia

In [4]:
# define a bounding box that contains only the area of Australia based on its WGS84 coordinates

coords = [112, -44, 154, -9]

df_aus = df_area[(df_area['longitude'] >= coords[0]) & (df_area['latitude'] >= coords[1]) & (df_area['longitude'] <= coords[2]) & (df_area['latitude'] <= coords[3])].copy()
df_aus.shape
df_aus.head(20)
df_aus.tail()

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
129908,-13.82279,126.98756,338.76,0.74,0.76,2026-05-06,610,N,VIIRS,n,2.0NRT,290.60,11.48,D
129909,-13.82120,126.99480,349.41,0.74,0.76,2026-05-06,610,N,VIIRS,n,2.0NRT,291.07,18.52,D
129910,-13.81871,126.98895,341.22,0.74,0.76,2026-05-06,610,N,VIIRS,n,2.0NRT,290.14,10.02,D
129911,-13.81864,126.80477,330.86,0.72,0.75,2026-05-06,610,N,VIIRS,n,2.0NRT,290.75,7.67,D
129912,-13.81711,126.99607,337.07,0.74,0.76,2026-05-06,610,N,VIIRS,n,2.0NRT,289.97,10.02,D


### Filter? for required Timeframe or add Datetime Column with active time 

In [18]:
# 1. 
# combine the acq_date and acq_time column to one acq_datetime column and set it to an active time format with pandas function to_datetime

## acq_date is a string in the format YYYY-MM_DD, 
## while acq_time is an integer in Greenwich Mean Time (e.g. 603 meaning 6:03), 
## so it needs to be converted to string too (with astype(str)),
## fill it up to 4 numbers with zeros, so that all times have the same length (with str.zfill(4), e.g. 603 -> 0603)
## and save it as the format '%Y-%m-%d %H%M'

###df_aus['acq_datetime'] = pd.to_datetime(df_aus['acq_date'] + ' ' + df_aus['acq_time'].astype(str).str.zfill(4), format='%Y-%m-%d %H%M')
###df_aus.head()

###print (f'Australia GMT timezone datetime value range: {df_aus['acq_datetime'].min()} to {df_aus['acq_datetime'].max()}')

# 2.
# convert GMT into local time?

# 3.
# # Set the timestamp column as the index ?
###hourly_data = hourly_data.set_index("timestamp")

# Notice how 'timestamp' drops down a level to become the index!
###display(hourly_data.head(3))



### Converting raw coordinates into geometries

In [5]:
# the projection EPSG:9473 is used for Australia, as it is recommended for national mapping

# convert latitude, longitude values into point geometry and set crs (since no crs extisting yet) with crs="EPSG:9473" to EPSG:9473

gdf_aus = gpd.GeoDataFrame(
    df_aus, geometry=gpd.points_from_xy(df_aus.longitude, df_aus.latitude), crs="EPSG:9473")
print(gdf_aus.crs)
gdf_aus.head()

EPSG:9473


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,geometry
1031,-43.17596,146.79762,333.68,0.53,0.42,2026-05-02,355,N,VIIRS,n,2.0NRT,277.32,4.22,D,POINT (146.798 -43.176)
1032,-42.90269,147.86391,338.48,0.48,0.40,2026-05-02,355,N,VIIRS,n,2.0NRT,271.66,6.20,D,POINT (147.864 -42.903)
1033,-42.87089,147.87529,335.38,0.48,0.40,2026-05-02,355,N,VIIRS,n,2.0NRT,271.61,3.57,D,POINT (147.875 -42.871)
1034,-42.78834,146.96037,334.01,0.52,0.41,2026-05-02,355,N,VIIRS,n,2.0NRT,281.36,7.59,D,POINT (146.96 -42.788)
1035,-42.78455,146.95894,351.57,0.52,0.41,2026-05-02,355,N,VIIRS,n,2.0NRT,282.52,7.59,D,POINT (146.959 -42.785)


## Group geometries by date with dissolve
This creates MultiPoint Geometries that we can later add to the map that we have one layer for each day

In [6]:
gdf_daily = gdf_aus.dissolve(by="acq_date")
gdf_daily

,geometry,latitude,longitude,bright_ti4,scan,track,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
acq_date,,,,,,,,,,,,,,
2026-05-02,"MULTIPOINT (114.674 -28.353, 114.698 -28.837, ...",-43.17596,146.79762,333.68,0.53,0.42,355,N,VIIRS,n,2.0NRT,277.32,4.22,D
2026-05-03,"MULTIPOINT (114.628 -28.153, 114.628 -28.155, ...",-37.90232,142.86154,345.11,0.43,0.62,338,N,VIIRS,n,2.0NRT,287.45,4.77,D
2026-05-04,"MULTIPOINT (114.167 -27.719, 114.17 -27.722, 1...",-41.80918,147.28889,351.19,0.47,0.64,317,N,VIIRS,n,2.0NRT,283.68,6.88,D
2026-05-05,"MULTIPOINT (114.122 -27.811, 114.139 -27.848, ...",-42.05627,147.78288,334.52,0.77,0.77,258,N,VIIRS,n,2.0NRT,284.28,7.22,D
2026-05-06,"MULTIPOINT (114.129 -27.806, 114.131 -27.816, ...",-9.93532,151.27454,329.41,0.51,0.66,249,N,VIIRS,n,2.0NRT,291.31,3.52,D


## Visualise it and create interactive Map

In [ ]:
import folium

# create basemap for the extent of Australia
aus_map = folium.Map(
    location=[-25.5649, 133.1234], # use the coordinates of Australia's centre (25°56′49.3″S, 133°12′34.7″E) for the location
    zoom_start=4,
    tiles="CartoDB Positron",  # clean, light basemap
)
# add the fire GeoDataFrame and custom markers to an orange flame
folium.GeoJson(
    gdf_daily,
    name="Wildfire",
    marker=folium.Marker(
        icon=folium.Icon(color="orange", icon="fire")
    ),).add_to(aus_map)

# add layers for each day
for date, row in gdf_daily.iterrows():
    fg = folium.FeatureGroup(name=str(date))
    
    folium.GeoJson(row.geometry).add_to(fg)
    
    fg.add_to(aus_map)

folium.LayerControl().add_to(aus_map)

# save the map (display does not work bc data file is too big)
aus_map.save("daily_fires_map2.html")

In [9]:
# alternative way: ?
##%pip install geodatasets cartopy
##from cartopy import crs as ccrs
##from geodatasets import get_path

##path = get_path("naturalearth.land")
##world = gpd.read_file(path)

##ax = world.plot(figsize=(10, 10), color="grey", edgecolor="black")
##ax.set_xlim([coords[0],  coords[2]])
##ax.set_ylim([coords[1],  coords[3]])